# Product Evidence Platform — Single-Product EDA and RCA

This notebook is the supported single-product runner **and** the complete post-run diagnostic report.

After updating only the credentials in `.env`, run:

```bash
./scripts/azureml_startup.sh
```

Then open this notebook and run the cells in order.

The notebook explains the complete funnel:

**SERP returned → unique candidates → scrape attempted → scrape successful → agentic browser investigated → identity accepted → requested-feature complete → finally selected**

It creates compact pandas tables, a Rich executive summary, stage/domain quality analysis, candidate-level acceptance RCA, feature evidence matrices, and graphical diagnostics using Matplotlib and Seaborn.

`COMPLETED` and `REVIEW_REQUIRED` are successful terminal workflow states. Only `FAILED` represents an execution failure.


In [ ]:
from __future__ import annotations

import importlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path
from pprint import pprint
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "docker-compose.yml").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root containing docker-compose.yml")


PROJECT_ROOT = find_project_root()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))


# The notebook installs only missing analytical packages into the active kernel.
# This makes a fresh Azure ML checkout runnable without a separate notebook setup step.
PACKAGE_IMPORTS = {
    "pandas": "pandas>=2.2,<3",
    "matplotlib": "matplotlib>=3.8,<4",
    "seaborn": "seaborn>=0.13.2,<1",
    "rich": "rich>=13.7,<15",
}
missing_specs = []
for module_name, package_spec in PACKAGE_IMPORTS.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing_specs.append(package_spec)

if missing_specs:
    print("Installing missing notebook diagnostics packages:", ", ".join(missing_specs))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_specs])
    importlib.invalidate_caches()

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rich.console import Console
from rich.panel import Panel

from product_evidence_harness.notebook_diagnostics import (
    build_single_product_diagnostics,
    display_compact,
    display_rich_summary,
    plot_all_diagnostics,
    plot_candidate_outcomes,
    plot_confidence_distribution,
    plot_confidence_vs_coverage,
    plot_domain_quality,
    plot_feature_heatmap,
    plot_funnel,
    plot_rejection_reasons,
    plot_stage_yield,
)

sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 180)

AGENT_URL = os.getenv("PRODUCT_AGENT_URL", "http://127.0.0.1:8788").rstrip("/")
POLL_SECONDS = 3
HEARTBEAT_SECONDS = 30
TERMINAL_STATUSES = {"COMPLETED", "REVIEW_REQUIRED", "FAILED"}
BOOTSTRAP_STATUS_PATH = PROJECT_ROOT / "data" / "runtime" / "stack_health.json"
DEFAULT_FEATURE_SET = "toy_features"


def api_json(method: str, path: str, payload: dict | None = None, timeout: int = 30) -> dict:
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{AGENT_URL}{path}",
        data=body,
        method=method,
        headers={"Content-Type": "application/json"},
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Agent API returned HTTP {exc.code}: {detail}\n"
            "Run ./scripts/azureml_startup.sh from the repository root and rerun this cell."
        ) from exc
    except URLError as exc:
        raise RuntimeError(
            f"Cannot reach {AGENT_URL}. Run ./scripts/azureml_startup.sh first."
        ) from exc


def available_feature_sets() -> list[str]:
    private_root = PROJECT_ROOT / "inputs" / "private"
    return sorted(path.stem for path in private_root.glob("*.json") if path.is_file())


def read_bootstrap_status() -> dict:
    if not BOOTSTRAP_STATUS_PATH.is_file():
        return {}
    return json.loads(BOOTSTRAP_STATUS_PATH.read_text(encoding="utf-8"))


def check_health() -> dict:
    health = api_json("GET", "/health", timeout=15)
    if health.get("status") != "healthy":
        raise RuntimeError(f"Platform is not healthy: {json.dumps(health, indent=2)}")
    configuration = health.get("configuration") or {}
    if not configuration.get("three_stage_contract_enforced"):
        raise RuntimeError("Agent is not running the strict three-stage contract")
    if configuration.get("serpapi_request_limit") != 3:
        raise RuntimeError("Agent search credit limit is not exactly 3")
    if not configuration.get("agentic_browser_contract_enforced"):
        raise RuntimeError("Agent is not running the required LLM-agentic browser contract")
    if not configuration.get("llm_configured"):
        raise RuntimeError("LLM configuration is not present")
    browser = health.get("browser_service") or {}
    if not browser.get("agentic_tools"):
        raise RuntimeError("Browser service does not expose agentic session tools")
    return health


def submit_product(product: dict, feature_set: str = DEFAULT_FEATURE_SET) -> str:
    response = api_json("POST", "/v1/jobs", {"product": product, "feature_set": feature_set})
    return response["job_id"]


def wait_for_job(
    job_id: str,
    poll_seconds: int = POLL_SECONDS,
    heartbeat_seconds: int = HEARTBEAT_SECONDS,
) -> dict:
    started = time.monotonic()
    last_signature = None
    last_printed_at = 0.0
    while True:
        status = api_json("GET", f"/v1/jobs/{job_id}", timeout=15)
        signature = (
            status.get("status"),
            status.get("stage"),
            status.get("message"),
        )
        now = time.monotonic()
        elapsed = int(now - started)
        if signature != last_signature or now - last_printed_at >= heartbeat_seconds:
            line = (
                f"{job_id}: {status['status']} | "
                f"{status.get('stage', '')} | {status.get('message', '')}"
            )
            if signature == last_signature:
                line += f" | still running ({elapsed}s elapsed)"
            print(line)
            last_signature = signature
            last_printed_at = now
        if status["status"] in TERMINAL_STATUSES:
            if status["status"] == "FAILED":
                raise RuntimeError(status.get("error") or status.get("message") or "Job failed")
            return status
        time.sleep(poll_seconds)


def run_product(product: dict, feature_set: str = DEFAULT_FEATURE_SET) -> dict:
    job_id = submit_product(product, feature_set)
    wait_for_job(job_id)
    return api_json("GET", f"/v1/jobs/{job_id}/result", timeout=60)


def host_artifact_dir(result: dict) -> Path | None:
    row_id = (result.get("product") or {}).get("row_id")
    return PROJECT_ROOT / "data" / "artifacts" / row_id if row_id else None


def summarize_result(result: dict) -> dict:
    product = result.get("product") or {}
    search = result.get("search") or {}
    agentic = result.get("agentic_browser") or {}
    product_match = result.get("product_match") or {}
    acceptance = result.get("primary_url_acceptance") or {}
    evidence_set = result.get("evidence_set") or {}
    return {
        "row_id": product.get("row_id"),
        "job_status": result.get("job_status"),
        "coding_ready": result.get("coding_ready"),
        "primary_url": result.get("primary_url"),
        "supplementary_urls": result.get("supplementary_urls") or [],
        "serpapi_requests_used": search.get("serpapi_requests_used"),
        "search_stages": [stage.get("name") for stage in search.get("stages") or []],
        "candidate_urls_admitted": agentic.get("candidate_urls_admitted"),
        "candidate_investigations_completed": agentic.get("candidate_investigations_completed"),
        "candidate_investigations_failed": agentic.get("candidate_investigations_failed"),
        "selection_scope": product_match.get("selection_scope"),
        "url_decision_status": product_match.get("url_decision_status"),
        "strict_primary_url_accepted": acceptance.get("accepted"),
        "acceptance_reasons": acceptance.get("reasons") or [],
        "evidence_status": evidence_set.get("status"),
        "coverage": evidence_set.get("total_coverage"),
        "covered_features": evidence_set.get("covered_features") or [],
        "missing_features": evidence_set.get("missing_features") or [],
        "conflicting_features": evidence_set.get("conflicting_features") or [],
        "host_artifact_dir": str(host_artifact_dir(result)),
    }


bootstrap_status = read_bootstrap_status()
health = check_health()
feature_sets = available_feature_sets()

if DEFAULT_FEATURE_SET not in feature_sets:
    raise RuntimeError(
        "The committed default inputs/private/toy_features.json is missing. "
        "Pull the latest branch and rerun ./scripts/azureml_startup.sh."
    )

Console().print(
    Panel(
        f"Repository: {PROJECT_ROOT}\n"
        f"Agent: {AGENT_URL}\n"
        f"Default feature set: {DEFAULT_FEATURE_SET}\n"
        f"Available feature sets: {', '.join(feature_sets)}",
        title="Notebook readiness",
    )
)
pprint(health)


## 1. Run one product

Update the product values below and set `RUN_SINGLE_PRODUCT = True`.

- `main_text` and `country_code` are mandatory.
- `retailer_name`, `ean`, and `language_code` are optional.
- Keep EAN/GTIN as text.
- `FEATURE_SET` already points to the committed `inputs/private/toy_features.json`.


In [ ]:
FEATURE_SET = "toy_features"
RUN_SINGLE_PRODUCT = False

product = {
    "row_id": "ROW-001",
    "main_text": "Replace with the exact product identity text",
    "country_code": "CH",
    "retailer_name": None,
    "ean": None,
    "language_code": None,
}

if RUN_SINGLE_PRODUCT:
    if product["main_text"].startswith("Replace with"):
        raise ValueError("Replace product['main_text'] before running")
    result = run_product(product, FEATURE_SET)
    pprint(summarize_result(result))
else:
    print("Ready. Replace the product input and set RUN_SINGLE_PRODUCT = True.")


## 2. Build the complete diagnostic model

This cell creates the main reusable tables:

- `results_df`: one row per deduplicated SERP candidate, enriched with scrape, agentic-browser, identity, feature-coverage, and final-selection status.
- `search_stages_df`: each SerpAPI credit/stage and its yield.
- `serp_results_df`: raw SERP rows when present; otherwise the persisted deduplicated candidate list.
- `funnel_df`: conversion from SERP output to final selection.
- `domain_summary_df`: domain-level quality and conversion.
- `rejection_reasons_df`: normalized candidate blockers and rejection reasons.
- `feature_evidence_df` and `feature_matrix_df`: URL-level feature evidence.
- `selection_rca_df`: the final root-cause analysis.


In [ ]:
if "result" not in globals():
    raise RuntimeError("Run the single-product cell first.")

artifact_path = host_artifact_dir(result)
if artifact_path is None or not artifact_path.is_dir():
    raise RuntimeError(
        "The run completed but its repository-local artifact directory was not found."
    )

diagnostics = build_single_product_diagnostics(
    result,
    artifact_dir=artifact_path,
)

overview_df = diagnostics.overview_df
search_stages_df = diagnostics.search_stages_df
serp_results_df = diagnostics.serp_results_df
results_df = diagnostics.results_df
agentic_df = diagnostics.agentic_df
feature_evidence_df = diagnostics.feature_evidence_df
feature_matrix_df = diagnostics.feature_matrix_df
funnel_df = diagnostics.funnel_df
domain_summary_df = diagnostics.domain_summary_df
stage_quality_df = diagnostics.stage_quality_df
rejection_reasons_df = diagnostics.rejection_reasons_df
selection_rca_df = diagnostics.selection_rca_df

display_rich_summary(diagnostics)
print(f"Artifact directory: {artifact_path}")
print(f"Candidate rows in results_df: {len(results_df)}")


## 3. Executive overview and final RCA

These compact tables answer:

- What was searched?
- How much evidence was obtained?
- Was a strict primary URL accepted?
- Why was the final URL chosen or rejected?


In [ ]:
display_compact(
    overview_df,
    title="Run overview",
    max_rows=30,
)
display_compact(
    selection_rca_df,
    title="Final URL selection RCA",
    max_rows=30,
)


## 4. SERP-stage EDA and conversion funnel

This section shows how many results each SerpAPI stage returned, how many were new, how many were scraped, and the conversion rate at every downstream gate.


In [ ]:
display_compact(
    search_stages_df,
    title="Three-stage SerpAPI execution",
    columns=[
        "serp_credit",
        "name",
        "scope",
        "language_code",
        "results_returned",
        "new_candidate_urls",
        "candidates_scraped",
        "query",
    ],
    max_rows=10,
)
display_compact(
    stage_quality_df,
    title="SERP stage quality ratios",
    columns=[
        "serp_credit",
        "name",
        "results_returned",
        "new_candidate_urls",
        "candidates_scraped",
        "result_to_new_candidate_rate",
        "new_candidate_to_scrape_rate",
    ],
    max_rows=10,
)
display_compact(
    funnel_df,
    title="Candidate conversion funnel",
    max_rows=20,
)


## 5. Candidate-level results (`results_df`)

`results_df` is the principal diagnostic table. It shows every retained candidate and the exact stage at which it passed or failed.

`quality_verified` means the deterministic candidate validation status is `VERIFIED`. It is not a subjective notebook score.


In [ ]:
RESULT_COLUMNS = [
    "candidate_rank",
    "stage",
    "domain",
    "serp_position",
    "confidence",
    "richness",
    "scrape_attempted",
    "scrape_success",
    "agentic_investigated",
    "agentic_status",
    "browser_openable",
    "text_scrapable",
    "rendered_product_verified",
    "identity_accepted",
    "coverage",
    "required_coverage",
    "critical_coverage",
    "feature_complete",
    "quality_verified",
    "strict_selected",
    "review_selected",
    "final_candidate_status",
    "missing_features_compact",
    "conflicts_compact",
    "url",
]

display_compact(
    results_df,
    title="Candidate-level acceptance and selection",
    columns=RESULT_COLUMNS,
    max_rows=50,
)


## 6. SERP candidate inventory

This is the URL inventory attributable to SERP discovery. When the runtime provides raw rows, duplicates and original positions remain visible. For earlier runs, the table falls back to the persisted deduplicated candidate inventory.


In [ ]:
display_compact(
    serp_results_df,
    title="SERP candidate inventory",
    columns=[
        "serp_credit",
        "stage",
        "scope",
        "position",
        "domain",
        "duplicate_url",
        "record_type",
        "title",
        "url",
    ],
    max_rows=100,
)


## 7. Agentic-browser investigation RCA

This table shows the LLM-controlled browser effort per candidate: status, number of reasoning turns, browser actions, termination reason, and errors.


In [ ]:
display_compact(
    agentic_df,
    title="Agentic-browser investigations",
    columns=[
        "candidate_id",
        "status",
        "turns_used",
        "actions_executed",
        "termination_reason",
        "domain",
        "requested_url",
        "final_url",
        "error",
    ],
    max_rows=100,
)


## 8. Domain quality and rejection analysis

The domain table helps explain whether SERP returned concentrated high-quality retailer evidence or many low-yield sources. The rejection table normalizes recurring blockers across deterministic validation, browser access, feature extraction, and agentic investigation.


In [ ]:
display_compact(
    domain_summary_df,
    title="Domain-level candidate quality",
    max_rows=30,
)
display_compact(
    rejection_reasons_df,
    title="Most frequent rejection and blocking reasons",
    max_rows=40,
)


## 9. Requested-feature evidence

The evidence table identifies the value, evidence status, extraction method, confidence, and source location for every URL-feature pair. The matrix gives a compact binary view of which URL supports which requested feature.


In [ ]:
display_compact(
    feature_evidence_df,
    title="Feature evidence by candidate URL",
    columns=[
        "domain",
        "identity_accepted",
        "identity_status",
        "feature_id",
        "feature_name",
        "value",
        "status",
        "supported",
        "confidence",
        "extraction_method",
        "evidence_location",
        "url",
    ],
    max_rows=100,
)

display_compact(
    feature_matrix_df.reset_index(),
    title="URL × requested-feature support matrix",
    max_rows=100,
)


## 10. Graphical diagnostics

Each figure answers a separate diagnostic question:

1. Where did candidate volume drop?
2. Which search stage produced usable candidates?
3. Why were candidates rejected?
4. How are confidence and feature coverage distributed?
5. Which domains contributed evidence?
6. Which requested features were supported by each URL?


In [ ]:
plot_funnel(diagnostics)
plot_stage_yield(diagnostics)
plot_candidate_outcomes(diagnostics)
plot_confidence_distribution(diagnostics)
plot_confidence_vs_coverage(diagnostics)
plot_domain_quality(diagnostics)
plot_rejection_reasons(diagnostics)
plot_feature_heatmap(diagnostics)


## 11. Export the diagnostic tables

This creates a reproducible Excel workbook containing every diagnostic table. The JSON result and original CSV artifacts remain the source-of-truth audit files.


In [ ]:
DIAGNOSTIC_WORKBOOK = artifact_path / "single_product_diagnostics.xlsx"

with pd.ExcelWriter(DIAGNOSTIC_WORKBOOK, engine="openpyxl") as writer:
    for name, frame in diagnostics.tables().items():
        safe_name = name.removesuffix("_df")[:31]
        frame.to_excel(writer, sheet_name=safe_name, index=name == "feature_matrix_df")

print(f"Diagnostic workbook written to: {DIAGNOSTIC_WORKBOOK}")


## 12. Raw audit artifacts

Use this only when an RCA requires the complete underlying JSON rather than the compact EDA tables.


In [ ]:
pprint(result.get("search") or {})
pprint(result.get("product_match") or {})
pprint(result.get("primary_url_acceptance") or {})
pprint(result.get("evidence_set") or {})

print("\nAvailable files:")
for path in sorted(artifact_path.rglob("*")):
    if path.is_file():
        print(path.relative_to(artifact_path))
